# Recovering ChIP-seq GRNs from homogeneous cell populations

scCAFM first predicts a **cell-specific GRN** for every cell. In a homogeneous population such as hESC or mESC, the cells share the same broad identity and condition, so they are expected to share a common regulatory program. We therefore average each TF–target score across all cells to obtain one **pooled GRN**.

This pooling is also what makes comparison with ChIP-seq appropriate. A ChIP-seq reference is a population-level measurement: it reports TF–DNA binding observed across many cells, rather than a separate network for each individual cell. Comparing the pooled scCAFM network with the ChIP-seq network therefore matches two population-level summaries.

Pooling reduces cell-to-cell fluctuation and highlights edges that are consistently strong across the population. It does **not** imply that every retained edge is active in every cell. For a heterogeneous dataset, one pooled GRN may mix distinct cell-type programs; in that setting, cells should be stratified first or examined with cell-specific GRNs.

## 1. Set up the tutorial

The model files are read from `assets/`. The four tutorial data files are read from `tutorial_data/chipseq_grn_recovery/`.

In [1]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import torch

from sccafm import (
    GRNInferencer,
    ScPreprocessor,
    evaluate_chipseq_grn,
    load_vocab_json,
    prepare_chipseq_reference,
    resolve_model_assets,
    write_pooled_grn_csv,
)


REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

MODEL_SOURCE = REPO_ROOT / "assets"
DATA_DIR = REPO_ROOT / "tutorial_data" / "chipseq_grn_recovery"

required_files = [
    DATA_DIR / "hESC.h5ad",
    DATA_DIR / "mESC.h5ad",
    DATA_DIR / "hESC-ChIP-seq.csv",
    DATA_DIR / "mESC-ChIP-seq.csv",
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Place the tutorial data files at the following paths:\n" + missing_text
    )

assets = resolve_model_assets(MODEL_SOURCE)
token_dict = load_vocab_json(assets.vocab)
print("Tutorial files are ready.")

Tutorial files are ready.


In [2]:
datasets = [
    {
        "name": "hESC",
        "species": "human",
        "dataset_path": DATA_DIR / "hESC.h5ad",
        "reference_path": DATA_DIR / "hESC-ChIP-seq.csv",
    },
    {
        "name": "mESC",
        "species": "mouse",
        "dataset_path": DATA_DIR / "mESC.h5ad",
        "reference_path": DATA_DIR / "mESC-ChIP-seq.csv",
    },
]

## 2. Prepare the expression data and ChIP-seq references

Each `.h5ad` file contains cells as rows and measured genes as columns. Both datasets contain normal embryonic stem cells from one species. Each ChIP-seq CSV contains directed reference edges:

- `Gene1`: regulator (TF);
- `Gene2`: target gene.

The table below is generated directly from the tutorial files. **Raw ChIP edges** is the number of rows in the reference. Before evaluation, scCAFM maps gene identifiers to its vocabulary, removes unmapped edges and self-edges, and collapses duplicate edges. **Mapped ChIP edges** is the resulting unique reference network. **Reference TFs** are mapped ChIP-seq regulators recognized by the species-specific TF catalogue before the expression data are filtered.

Some reference TFs may not be measured in a dataset. The final results therefore report **Retained TFs**: reference TFs that remain in the processed expression data and act as GRN sources. Candidate edges connect each retained TF to every retained target gene except itself, so their number is `Retained TFs × (Genes − 1)`.

During evaluation, mapped ChIP-seq edges are positive examples. Other non-self edges from a retained TF to a retained gene form the remaining candidate set.

For preprocessing, we preserve every reference TF that is measured and passes gene filtering, then select additional variable non-TF genes using the `n_top_genes` setting below. Other catalogue TFs are excluded from HVG selection so that the inferred source set matches the retained reference regulators.

In [3]:
references = {}
overview_rows = []

for item in datasets:
    reference = prepare_chipseq_reference(
        item["reference_path"],
        model_source=MODEL_SOURCE,
        species=item["species"],
    )
    references[item["name"]] = reference

    dataset_info = sc.read_h5ad(item["dataset_path"], backed="r")
    condition = ", ".join(
        sorted(dataset_info.obs["disease"].astype(str).unique().tolist())
    )
    overview_rows.append(
        {
            "Dataset": item["name"],
            "Species": item["species"],
            "Condition": condition,
            "Cells": dataset_info.n_obs,
            "Measured genes": dataset_info.n_vars,
            "Raw ChIP edges": reference.raw_edge_count,
            "Mapped ChIP edges": reference.mapped_edge_count,
            "Unmapped edges": reference.unmapped_edge_count,
            "Self-edges removed": reference.self_loop_edge_count,
            "Duplicate edges": reference.duplicate_edge_count,
            "Reference TFs": len(reference.supported_tfs),
        }
    )
    dataset_info.file.close()

dataset_overview = pd.DataFrame(overview_rows)
dataset_overview.style.format(
    {
        "Cells": "{:,.0f}",
        "Measured genes": "{:,.0f}",
        "Raw ChIP edges": "{:,.0f}",
        "Mapped ChIP edges": "{:,.0f}",
        "Unmapped edges": "{:,.0f}",
        "Self-edges removed": "{:,.0f}",
        "Duplicate edges": "{:,.0f}",
        "Reference TFs": "{:,.0f}",
    }
)

,Dataset,Species,Condition,Cells,Measured genes,Raw ChIP edges,Mapped ChIP edges,Unmapped edges,Self-edges removed,Duplicate edges,Reference TFs
0,hESC,human,normal,758,"17,735","441,671","423,186","18,423",62,0,108
1,mESC,mouse,normal,421,"18,385","985,381","821,698","163,585",98,0,147


In [4]:
prepared = {}

for item in datasets:
    reference = references[item["name"]]
    raw_adata = sc.read_h5ad(item["dataset_path"])
    raw_shape = raw_adata.shape

    preprocessor = ScPreprocessor(
        min_genes=200,
        min_cells=3,
        max_pct_counts_mt=20.0,
        target_sum=10000,
        log1p=False,
        n_top_genes=2000,
        hvg_flavor="seurat",
        subset_hvg=True,
        remove_mito_genes=True,
        remove_ribo_genes=True,
        remove_hb_genes=True,
        token_dict=token_dict,
        preserve_gene_names=list(reference.supported_tfs),
        hvg_exclude_gene_names=list(reference.catalog_tfs),
        hvg_exclude_preserved_genes=True,
        sanitize_X=True,
        inplace=False,
    )
    adata = preprocessor(raw_adata)
    prepared[item["name"]] = {
        "adata": adata,
        "reference": reference,
        "raw_shape": raw_shape,
    }

    print(
        f"{item['name']}: {raw_shape[0]:,} → {adata.n_obs:,} cells; "
        f"{raw_shape[1]:,} → {adata.n_vars:,} genes"
    )

hESC: 758 → 758 cells; 17,735 → 2,108 genes
mESC: 421 → 421 cells; 18,385 → 2,139 genes


## 3. Infer and evaluate pooled GRNs

The model is loaded once on a single GPU. This tutorial explicitly uses `attention_backend="fa4"`, the default high-performance attention backend used by the current scCAFM model. FA4 changes how attention is computed efficiently on supported GPUs; it does not change the definition of the inferred GRN. If FA4 is unavailable on a GPU, use `attention_backend="fa2"` when the FA2 kernels are installed.

We do not apply a score threshold or top-k filter before evaluation, so all raw candidate scores are used.

In [5]:
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires one CUDA GPU.")

inferencer = GRNInferencer.from_pretrained(
    MODEL_SOURCE,
    device="cuda:0",
    attention_backend="fa4", # or "fa2" if fa4 is not available
    max_length=2560,
    species_key="species",
    disease_key="disease",
)
print("scCAFM is ready on cuda:0.")

scCAFM is ready on cuda:0.


In [6]:
pooled_grns = {}
rows = []

for item in datasets:
    name = item["name"]
    payload = prepared[name]
    pooled_grn = inferencer.infer_pooled(
        payload["adata"],
        batch_size=8,
        score_threshold=None,
        top_k_edges=None,
    )
    evaluation = evaluate_chipseq_grn(pooled_grn, payload["reference"])
    pooled_grns[name] = pooled_grn

    retained_tfs = len(pooled_grn.source_genes)
    expected_candidates = retained_tfs * (pooled_grn.shape[1] - 1)
    if evaluation.n_candidates != expected_candidates:
        raise RuntimeError(
            f"Candidate-edge count is inconsistent for {name}: "
            f"{evaluation.n_candidates:,} != {retained_tfs:,} x "
            f"({pooled_grn.shape[1]:,} - 1)."
        )

    rows.append(
        {
            "Dataset": name,
            "Cells": pooled_grn.n_cells,
            "Genes": pooled_grn.shape[1],
            "Retained TFs": retained_tfs,
            "Candidate edges": evaluation.n_candidates,
            "Positive edges": evaluation.n_positive_edges,
            "AUPRC": evaluation.auprc,
            "Early precision": evaluation.early_precision,
        }
    )
    print(f"Finished {name}.")

summary = pd.DataFrame(rows)

/data1021/xukaichen/miniconda3/envs/py312/lib/python3.12/site-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


Finished hESC.
Finished mESC.


### Optional: save all, top-ranked, or thresholded edges

The evaluation above uses the complete, unfiltered pooled GRN. For export, you may instead save:

- `"all"`: every TF–target edge;
- `"top_k"`: the globally top-ranked edges across the complete pooled TF-by-gene matrix;
- `"threshold"`: edges with `score >= SCORE_THRESHOLD`.

Top-k and threshold filtering are mutually exclusive. A threshold is a user-chosen score cutoff, not a statistical significance level, so there is no universal value that is suitable for every dataset. Filtered exports rerun pooled inference and write only the retained edges.

In [7]:
SAVE_POOLED_GRNS = False
EDGE_SELECTION = "all"  # Choose "all", "top_k", or "threshold".
TOP_K_EDGES = 10000
SCORE_THRESHOLD = 0.1

if SAVE_POOLED_GRNS:
    if EDGE_SELECTION not in {"all", "top_k", "threshold"}:
        raise ValueError("EDGE_SELECTION must be 'all', 'top_k', or 'threshold'.")

    output_dir = REPO_ROOT / "results"
    output_dir.mkdir(parents=True, exist_ok=True)

    for item in datasets:
        name = item["name"]
        if EDGE_SELECTION == "all":
            export_grn = pooled_grns[name]
        else:
            export_grn = inferencer.infer_pooled(
                prepared[name]["adata"],
                batch_size=8,
                score_threshold=(
                    SCORE_THRESHOLD if EDGE_SELECTION == "threshold" else None
                ),
                top_k_edges=TOP_K_EDGES if EDGE_SELECTION == "top_k" else None,
            )

        write_pooled_grn_csv(
            export_grn,
            output_dir / f"{name}_pooled_grn_{EDGE_SELECTION}.csv",
            overwrite=True,
        )

    print(f"Saved {EDGE_SELECTION} pooled GRNs to {output_dir}")

## Results

**AUPRC** summarizes how well the complete ranking recovers ChIP-seq edges. **Early precision** measures the fraction of supported edges among the highest-ranked predictions; higher values are better.

In [8]:
summary.style.format(
    {
        "Cells": "{:,.0f}",
        "Genes": "{:,.0f}",
        "Retained TFs": "{:,.0f}",
        "Candidate edges": "{:,.0f}",
        "Positive edges": "{:,.0f}",
        "AUPRC": "{:.4f}",
        "Early precision": "{:.4f}",
    }
)

,Dataset,Cells,Genes,Retained TFs,Candidate edges,Positive edges,AUPRC,Early precision
0,hESC,758,"2,108",108,"227,556","49,315",0.4258,0.4116
1,mESC,421,"2,139",139,"297,182","77,719",0.4497,0.4028
